# Cost-Aware LLM Router

LLM costs vary by four orders of magnitude across models. A query that needs a one-word factual answer should never be sent to GPT-4o. This notebook builds a pipeline that classifies query complexity, routes each query to the cheapest suitable model, enforces hard budget limits, and handles provider outages with a circuit breaker.

**What you'll build:** A pipeline that uses Groq for simple queries (~$0.000001), GPT-4o for complex ones (~$0.03), tracks costs in nested scopes, and automatically trips open when a provider fails.

**Time:** ~20 min | **Difficulty:** Intermediate

**Pricing reference (2026):**

| Model | Input (per 1M tokens) | Output (per 1M tokens) | Best for |
|---|---|---|---|
| Groq / llama-3.1-8b-instant | $0.05 | $0.08 | Simple lookups, classification |
| Groq / llama-3.3-70b-versatile | $0.59 | $0.79 | Medium reasoning |
| GPT-4o-mini | $0.15 | $0.60 | General Q&A |
| GPT-4o | $2.50 | $10.00 | Complex analysis, code |

## Prerequisites & Setup

In [ ]:
!pip install "synapsekit[openai,groq,evaluation]" -q

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["GROQ_API_KEY"] = "gsk_..."

## Step 1: Set up providers and cost tracking

`CostTracker` is the central accumulator. `BudgetGuard` wraps it and raises `BudgetExceededError` on limit breach.

In [ ]:
import asyncio
from synapsekit import CostTracker, BudgetGuard, BudgetLimit, LLMConfig
from synapsekit.llms.openai import OpenAILLM
from synapsekit.llms.groq import GroqLLM

# CostTracker is the central accumulator.
# It records every LLM call and converts token counts to USD.
tracker = CostTracker()

# BudgetGuard wraps CostTracker and raises BudgetExceededError on limit breach.
# per_request=$0.10 prevents a single complex query from burning through budget.
# daily=$50.00 caps total daily spend across all requests.
guard = BudgetGuard(
    BudgetLimit(per_request=0.10, daily=50.00),
    tracker=tracker    # Attach so guard can check the running daily total
)

# Groq: ultra-fast inference, open-source models, very cheap.
groq_cheap = GroqLLM(
    model="llama-3.1-8b-instant",
    config=LLMConfig(temperature=0.1, max_tokens=512)
)

groq_medium = GroqLLM(
    model="llama-3.3-70b-versatile",
    config=LLMConfig(temperature=0.2, max_tokens=1024)
)

# OpenAI GPT-4o: best reasoning quality, highest cost.
gpt4o = OpenAILLM(
    model="gpt-4o",
    config=LLMConfig(temperature=0.2, max_tokens=2048)
)

gpt4o_mini = OpenAILLM(
    model="gpt-4o-mini",
    config=LLMConfig(temperature=0.1, max_tokens=1024)
)

print("Providers and cost tracking configured.")

## Step 2: Build the complexity classifier

Use the cheapest model to classify query complexity. This meta-call costs ~$0.000001 — negligible compared to the savings from routing simple queries away from GPT-4o.

In [ ]:
from enum import Enum

class Complexity(Enum):
    SIMPLE   = "simple"    # factual lookup, single-step, no reasoning required
    MEDIUM   = "medium"    # multi-step reasoning, some domain knowledge
    COMPLEX  = "complex"   # code generation, deep analysis, long-form synthesis

# Routing table: complexity -> (llm, model_name_for_tracking)
ROUTING_TABLE = {
    Complexity.SIMPLE:  (groq_cheap,  "groq/llama-3.1-8b-instant"),
    Complexity.MEDIUM:  (groq_medium, "groq/llama-3.3-70b-versatile"),
    Complexity.COMPLEX: (gpt4o,       "gpt-4o"),
}

async def classify_complexity(query: str) -> Complexity:
    """Use the cheapest model to classify query complexity."""
    prompt = f"""Classify the complexity of the following query as simple, medium, or complex.

simple  = factual lookup, single answer, no reasoning (e.g. "What year was Python created?")
medium  = multi-step, some reasoning (e.g. "Explain the difference between TCP and UDP")
complex = deep analysis, code generation, or long synthesis (e.g. "Write a B-tree implementation")

Reply with exactly one word: simple, medium, or complex.

Query: {query}"""

    with tracker.scope("classify"):
        response = await groq_cheap.agenerate(prompt)
        rec = tracker.record("groq/llama-3.1-8b-instant", input_tokens=80, output_tokens=1)

    label = response.text.strip().lower()
    mapping = {
        "simple":  Complexity.SIMPLE,
        "medium":  Complexity.MEDIUM,
        "complex": Complexity.COMPLEX,
    }
    complexity = mapping.get(label, Complexity.MEDIUM)  # Default to medium on unexpected output
    print(f"  Classified as: {complexity.value}  (classifier cost: ${rec.cost_usd:.7f})")
    return complexity

print("Complexity classifier defined.")

## Step 3: Nested cost scopes

`CostTracker` supports a scope hierarchy so you can attribute costs at any level — by pipeline run, by step, or by individual LLM call.

In [ ]:
async def run_pipeline(query: str, pipeline_id: str = "default") -> dict:
    """Route a query through the cost-aware pipeline with nested scope tracking."""

    with tracker.scope(f"pipeline:{pipeline_id}"):

        # Raises BudgetExceededError if today's spend already hit the daily limit.
        guard.check_before(estimated_cost=0)

        print(f"\nQuery: {query!r}")
        with tracker.scope("step:classify"):
            complexity = await classify_complexity(query)

        # Route to the appropriate LLM
        llm, model_name = ROUTING_TABLE[complexity]
        print(f"  Routing to: {model_name}")

        with tracker.scope(f"step:answer"), tracker.scope(f"llm:{model_name}"):
            response = await llm.agenerate(query)

            # Record actual token usage
            rec = tracker.record(
                model_name,
                input_tokens=len(query.split()) * 2,
                output_tokens=len(response.text.split())
            )

        # Raises BudgetExceededError if this single request exceeded per_request limit.
        guard.check_after(rec.cost_usd)

    pipeline_cost = tracker.scope_cost(f"pipeline:{pipeline_id}")
    print(f"  Answer cost: ${rec.cost_usd:.7f}  |  Pipeline total: ${pipeline_cost:.7f}")
    return {
        "query":      query,
        "complexity": complexity.value,
        "model":      model_name,
        "answer":     response.text,
        "cost_usd":   rec.cost_usd,
    }

print("run_pipeline() defined.")

## Step 4: Circuit breaker

The circuit breaker prevents cascading failures when a provider is down. It transitions through three states:

```
CLOSED --(N failures in window)--> OPEN --(timeout)--> HALF_OPEN --(success)--> CLOSED
                                                                 --(failure)--> OPEN
```

When the breaker is OPEN it raises `CircuitBreakerOpen` immediately without making any network call — saving both latency and cost during an outage.

In [ ]:
from synapsekit.resilience import CircuitBreaker, CircuitBreakerOpen

# CircuitBreaker trips OPEN after 3 failures within 30 seconds.
gpt4o_breaker = CircuitBreaker(
    name="gpt-4o",
    failure_threshold=3,
    window_seconds=30,
    recovery_timeout=60,
)

groq_breaker = CircuitBreaker(
    name="groq",
    failure_threshold=5,
    window_seconds=60,
    recovery_timeout=30,
)

async def resilient_generate(query: str, complexity: Complexity) -> tuple[str, str]:
    """Generate an answer, falling back across providers if a circuit is open."""
    primary_llm, primary_model = ROUTING_TABLE[complexity]

    if complexity == Complexity.COMPLEX:
        try:
            async with gpt4o_breaker:
                response = await primary_llm.agenerate(query)
                return response.text, primary_model
        except CircuitBreakerOpen:
            print("  [circuit] gpt-4o OPEN -- falling back to gpt-4o-mini")
            async with groq_breaker:
                response = await gpt4o_mini.agenerate(query)
                return response.text, "gpt-4o-mini"
    else:
        try:
            async with groq_breaker:
                response = await primary_llm.agenerate(query)
                return response.text, primary_model
        except CircuitBreakerOpen:
            print(f"  [circuit] groq OPEN -- falling back to gpt-4o-mini")
            response = await gpt4o_mini.agenerate(query)
            return response.text, "gpt-4o-mini"

print("Circuit breaker defined.")

## Step 5: Write eval cases with cost thresholds

`@eval_case` with `MaxCostUSD` ensures your pipeline stays within budget in CI.

In [ ]:
# Eval cases (save as tests/test_cost_pipeline.py and run with `synapsekit test`)
from synapsekit.evaluation import eval_case, EvalSuite
from synapsekit.evaluation.metrics import MaxCostUSD, ContainsKeywords

@eval_case(description="Simple query routes to Groq and costs under $0.001")
async def test_simple_query_cheap():
    result = await run_pipeline("What year was Python first released?", "eval-simple")
    return {
        "output":     result["answer"],
        "cost_usd":   result["cost_usd"],
        "complexity": result["complexity"],
    }

@eval_case(description="Daily budget guard blocks requests after limit is hit")
async def test_budget_guard_blocks():
    from synapsekit import BudgetExceededError
    # Set a limit so small it will always trip
    tight_guard = BudgetGuard(BudgetLimit(per_request=0.000001))
    try:
        tight_guard.check_before(estimated_cost=0.10)
        return {"blocked": False}
    except BudgetExceededError:
        return {"blocked": True}

suite = EvalSuite(
    cases=[test_simple_query_cheap, test_budget_guard_blocks],
    metrics=[
        MaxCostUSD(threshold=0.001, apply_to=["test_simple_query_cheap"]),
    ]
)

print("Eval suite defined. Run with: synapsekit test tests/test_cost_pipeline.py")

## Complete Working Example

A self-contained demo wrapped in `asyncio.run(main())`.

In [ ]:
import asyncio
from synapsekit import CostTracker, BudgetGuard, BudgetLimit, LLMConfig
from synapsekit.llms.openai import OpenAILLM
from synapsekit.llms.groq import GroqLLM
from synapsekit.resilience import CircuitBreaker, CircuitBreakerOpen

async def main():
    tracker    = CostTracker()
    guard      = BudgetGuard(BudgetLimit(per_request=0.10, daily=5.00), tracker=tracker)
    groq_llm   = GroqLLM(model="llama-3.1-8b-instant", config=LLMConfig(temperature=0.1))
    openai_llm = OpenAILLM(model="gpt-4o-mini",         config=LLMConfig(temperature=0.2))
    breaker    = CircuitBreaker("openai", failure_threshold=3, recovery_timeout=30)

    queries = [
        ("simple",  "What does HTTP stand for?"),
        ("simple",  "Who wrote Romeo and Juliet?"),
        ("complex", "Explain transformer self-attention with a code example in Python."),
        ("medium",  "What are the trade-offs between microservices and monolithic architecture?"),
    ]

    print("=== Cost-Aware Pipeline Demo ===\n")
    for complexity_hint, query in queries:
        print(f"Query [{complexity_hint}]: {query[:60]}...")

        with tracker.scope(f"query:{complexity_hint}"):
            guard.check_before(0)

            if complexity_hint in ("simple", "medium"):
                response = await groq_llm.agenerate(query)
                model    = "groq/llama-3.1-8b-instant"
                in_tok, out_tok = 50, 80
            else:
                try:
                    async with breaker:
                        response = await openai_llm.agenerate(query)
                        model    = "gpt-4o-mini"
                        in_tok, out_tok = 120, 400
                except CircuitBreakerOpen:
                    print("  [circuit] OpenAI OPEN -- skipping complex query")
                    continue

            rec = tracker.record(model, input_tokens=in_tok, output_tokens=out_tok)
            guard.check_after(rec.cost_usd)

        print(f"  Model:  {model}")
        print(f"  Answer: {response.text[:80]}...")
        print(f"  Cost:   ${rec.cost_usd:.7f}\n")

    print("=== Final Cost Summary ===")
    print(tracker.summary())

asyncio.run(main())

## Next Steps

- **[LLM Fallback Chains](fallback-chain.ipynb)** — automatic failover without manual circuit breaker wiring
- **[Cost tracker reference](../../observability/cost-tracker)** — full `CostTracker` and `BudgetGuard` API
- **[Evaluation overview](../../evaluation/overview)** — `@eval_case`, `EvalSuite`, running with `synapsekit test`

**Troubleshooting:**
- `BudgetExceededError` during testing: Create a separate `CostTracker` instance per test to avoid accumulation
- Circuit breaker stays OPEN indefinitely: Use `recovery_timeout=5` in development
- Cost numbers look wrong: Use `tiktoken` for precise token counting, or set `track_tokens=True` in `LLMConfig`